# 03. Результаты: качество против памяти и времени

Ноутбук читает `results/*/` (конфиги, `train_metrics.json`, `eval.json`, `log_history.json`), ничего не пересчитывает и строит:

1. Сводную таблицу основных методов (mean ± std по seed)
2. **Качество против пиковой памяти и времени обучения** (главный график)
3. Ресурсы: память, скорость, число обучаемых параметров
4. Забывание общих навыков (проверка H5)
5. Кривые обучения
6. Подбор скорости обучения (обоснование выбранного lr)
7. Влияние размера выборки (H3) и ранга LoRA (H4)
8. Числа для проверки гипотез H1–H5

Графики сохраняются в `results/figures/`, таблица в `results/main_table.md` (вставляется в README).

In [ ]:
import os, sys, json, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

# корень репозитория: ноутбук можно запускать и из notebooks/, и из корня
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

FIG = ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

from src.aggregate import flatten_eval

NAMES = {"baseline": "Исходная модель", "full": "Полный FT", "lora": "LoRA (fp16)", "qlora": "QLoRA (NF4)"}
COL = {"baseline": "#7f7f7f", "full": "#1f77b4", "lora": "#2ca02c", "qlora": "#d62728"}
METHODS = ["full", "lora", "qlora"]

# колонки метрик из lm-eval (ключ: "задача|шоты/метрика"); при другой версии lm-eval проверьте имена в runs.columns
M_GSM = "gsm8k|0shot/exact_match,flexible-extract"
M_GSM_STRICT = "gsm8k|0shot/exact_match,strict-match"
M_GSM_5 = "gsm8k|5shot/exact_match,flexible-extract"
BENCH = {
    "ARC-Easy": "arc_easy|0shot/acc,none",
    "ARC-Challenge": "arc_challenge|0shot/acc_norm,none",
    "HellaSwag": "hellaswag|0shot/acc_norm,none",
    "MMLU": "mmlu|0shot/acc,none",
}
N_TEST = 1319   # размер GSM8K test

## Загрузка прогонов

Каждый каталог `results/<run>/` это один прогон. Выбранные скорости обучения читаются из `configs/method/*.yaml`: в «основной» набор попадают только прогоны с ними, а остальные (из lr-sweep) идут в раздел 6.
Прогоны, продолженные с чекпоинта (`resumed_from_checkpoint=True`), имеют неполные время и память, их нужно исключить из графиков ресурсов.

In [ ]:
def load_runs():
    rows = []
    for d in sorted((ROOT / "results").iterdir()):
        cp = d / "config.yaml"
        if not cp.exists():
            continue
        cfg = yaml.safe_load(cp.read_text())
        method = cfg.get("method")
        row = dict(run=d.name, method=method, dataset=cfg["data"]["train_dataset"],
                   train_size=cfg["data"].get("train_size"), r=(cfg.get("lora") or {}).get("r"),
                   lr=np.nan if method == "baseline" else float(cfg["train"]["lr"]),
                   seed=cfg.get("seed"), epochs=cfg.get("train", {}).get("epochs"))
        tp = d / "train_metrics.json"
        if tp.exists():
            tm = json.loads(tp.read_text())
            for k in ("trainable_params", "total_params", "peak_mem_allocated_gb", "wall_time_s",
                      "tokens_per_s", "final_val_loss", "resumed_from_checkpoint"):
                row[k] = tm.get(k)
            row["n_train"] = tm.get("data", {}).get("n_train")
        ep = d / "eval.json"
        if ep.exists():
            row.update(flatten_eval(json.loads(ep.read_text())))
        rows.append(row)
    df = pd.DataFrame(rows)
    num = ["train_size", "r", "lr", "seed", "epochs", "trainable_params", "total_params", "peak_mem_allocated_gb",
           "wall_time_s", "tokens_per_s", "final_val_loss", "n_train"]
    for c in num:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df["wall_min"] = df.wall_time_s / 60
    df["trainable_pct"] = 100 * df.trainable_params / df.total_params
    return df

runs = load_runs()
print(len(runs), "прогонов:", runs.method.value_counts().to_dict())

resumed = runs[runs.get("resumed_from_checkpoint") == True]
if len(resumed):
    print("ВНИМАНИЕ: продолжены с чекпоинта (время и память неполные):", resumed.run.tolist())
    runs.loc[resumed.index, ["wall_min", "wall_time_s", "tokens_per_s", "peak_mem_allocated_gb"]] = np.nan

missing = [c for c in [M_GSM, *BENCH.values()] if c not in runs.columns]
if missing:
    print("Нет колонок (возможно, ещё не запущена оценка или другая версия lm-eval):", missing)

In [ ]:
chosen = {m: float(yaml.safe_load((ROOT / f"configs/method/{m}.yaml").read_text())["train"]["lr"]) for m in METHODS}
R_CORE = yaml.safe_load((ROOT / "configs/method/lora.yaml").read_text())["lora"]["r"]
print("выбранные lr:", chosen, "| основной ранг:", R_CORE)

ft = runs[runs.method != "baseline"].copy()
ft["lr_chosen"] = ft.apply(lambda x: np.isclose(x.lr, chosen[x.method]), axis=1)

# весь GSM8K train, выбранный lr: основные + варианты рангов
full_data = ft[(ft.dataset == "gsm8k") & ft.train_size.isna() & ft.lr_chosen]
core = full_data[(full_data.method == "full") | (full_data.r == R_CORE)]
base = runs[runs.method == "baseline"]
print("основной набор:", core.groupby("method").size().to_dict(), "| baseline:", len(base))

## 1. Сводная таблица (основной набор)

mean ± std по seed (std по трём seed это грубая оценка, но честная). Строка «Исходная модель» это базовое решение из 02.

In [ ]:
def fmt(s, d=2, scale=1.0):
    s = s.dropna() * scale
    if len(s) == 0:
        return "—"
    return f"{s.mean():.{d}f}" + (f" ± {s.std(ddof=1):.{d}f}" if len(s) > 1 else "")

COLS = {
    "GSM8K flexible, %": (M_GSM, 2, 100),
    "GSM8K strict, %": (M_GSM_STRICT, 2, 100),
    "val loss": ("final_val_loss", 3, 1),
    "пик. память, ГБ": ("peak_mem_allocated_gb", 2, 1),
    "время, мин": ("wall_min", 1, 1),
    "токенов/с": ("tokens_per_s", 0, 1),
    "обучаемых, %": ("trainable_pct", 2, 1),
}
rows = []
for m in ["baseline", *METHODS]:
    d = base if m == "baseline" else core[core.method == m]
    row = {"Метод": NAMES[m], "seed": len(d)}
    for name, (c, dec, sc) in COLS.items():
        row[name] = fmt(d[c], dec, sc) if c in d.columns else "—"
    rows.append(row)
table = pd.DataFrame(rows).set_index("Метод")

def to_md(df):
    head = [df.index.name or ""] + list(df.columns)
    lines = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)]
    for idx, r in df.iterrows():
        lines.append("| " + " | ".join([str(idx)] + [str(v) for v in r]) + " |")
    return "\n".join(lines)

(ROOT / "results" / "main_table.md").write_text(to_md(table))
table

In [ ]:
if M_GSM in core and len(core):
    p = core[M_GSM].mean()
    ci = 1.96 * np.sqrt(p * (1 - p) / N_TEST) * 100
    ci_diff = 1.96 * np.sqrt(2 * p * (1 - p) / N_TEST) * 100
    print(f"Один прогон: 95% ДИ ≈ ±{ci:.1f} п.п. при точности {100*p:.1f}%")
    print(f"Разность двух независимых прогонов различима, если больше ≈ {ci_diff:.1f} п.п. (парное сравнение на тех же задачах немного чувствительнее)")

**Как читать.** Если разница между методами меньше, чем разброс по seed и доверительный интервал выше, её нельзя интерпретировать как «метод A лучше B». Это главная защита от неверных выводов в H1.

## 2. Качество против памяти и времени

Главный график работы. Крупные маркеры это среднее по seed с полосами std, бледные точки это отдельные seed. Пунктир это исходная модель (0-shot и, если есть, 5-shot).
Лучший результат: **выше и левее**. Верхний ряд в общем масштабе (с исходной моделью), нижний приближает разницу между дообученными методами, которая в общем масштабе почти не видна.

In [ ]:
def baseline_lines(ax):
    if len(base) and M_GSM in base:
        ax.axhline(100 * base[M_GSM].mean(), color=COL["baseline"], ls="--", lw=1, label="исходная, 0-shot")
    if len(base) and M_GSM_5 in base and base[M_GSM_5].notna().any():
        ax.axhline(100 * base[M_GSM_5].mean(), color=COL["baseline"], ls=":", lw=1.2, label="исходная, 5-shot")

def tradeoff(ax, df, xcol, xlabel, xscale=1.0, label_rank=False, zoom=False):
    d0 = df.assign(rr=df.r.fillna(0))
    for (m, rr), d in d0.groupby(["method", "rr"]):
        d = d.dropna(subset=[xcol, M_GSM])
        if d.empty:
            continue
        x, y = d[xcol] * xscale, d[M_GSM] * 100
        sx = x.std(ddof=1) if len(d) > 1 else 0.0
        sy = y.std(ddof=1) if len(d) > 1 else 0.0
        ax.scatter(x, y, color=COL[m], alpha=0.3, s=18)
        ax.errorbar(x.mean(), y.mean(), xerr=sx, yerr=sy, fmt="o", ms=9, capsize=4, color=COL[m],
                    label=NAMES[m] if rr in (0, R_CORE) else None)
        if label_rank and rr > 0:
            ax.annotate(f"r={int(rr)}", (x.mean(), y.mean()), textcoords="offset points", xytext=(6, 5), fontsize=8)
    if not zoom:
        baseline_lines(ax)
    ax.set(xlabel=xlabel, ylabel="GSM8K, flexible, %")

fig, axes = plt.subplots(2, 2, figsize=(12, 8.5), sharey="row")
tradeoff(axes[0, 0], core, "peak_mem_allocated_gb", "пиковая память GPU при обучении, ГБ")
tradeoff(axes[0, 1], core, "wall_min", "время обучения, мин")
tradeoff(axes[1, 0], core, "peak_mem_allocated_gb", "пиковая память GPU при обучении, ГБ", zoom=True)
tradeoff(axes[1, 1], core, "wall_min", "время обучения, мин", zoom=True)
axes[0, 0].set_title("Качество против памяти")
axes[0, 1].set_title("Качество против времени")
axes[1, 0].set_title("То же, масштаб только по дообученным моделям")
axes[1, 1].set_title("То же, масштаб только по дообученным моделям")
axes[0, 0].legend(loc="best", fontsize=8)
plt.tight_layout()
plt.savefig(FIG / "quality_vs_memory_time.png", bbox_inches="tight")
plt.show()

Тот же график для всех рангов LoRA/QLoRA (если стадия `rank` посчитана): видно, как «покупается» качество памятью и временем. Подписи это ранг; для этих точек обычно один seed, поэтому полос нет.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
tradeoff(axes[0], full_data, "peak_mem_allocated_gb", "пиковая память GPU, ГБ", label_rank=True)
tradeoff(axes[1], full_data, "wall_min", "время обучения, мин", label_rank=True)
axes[0].set_title("Все конфигурации: память")
axes[1].set_title("Все конфигурации: время")
axes[0].legend(loc="best", fontsize=8)
plt.tight_layout()
plt.savefig(FIG / "quality_vs_memory_time_all_ranks.png", bbox_inches="tight")
plt.show()

## 3. Ресурсы

Память, скорость обучения (токены в секунду; выше лучше) и доля обучаемых параметров (логарифмическая шкала). У QLoRA память меньше, но шаг обычно медленнее из-за деквантизации весов: это проверка H2.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
specs = [("peak_mem_allocated_gb", "пиковая память, ГБ", False),
         ("tokens_per_s", "токенов в секунду", False),
         ("trainable_pct", "обучаемых параметров, %", True)]
for ax, (c, title, log) in zip(axes, specs):
    for i, m in enumerate(METHODS):
        s = core[core.method == m][c].dropna()
        if s.empty:
            continue
        ax.bar(i, s.mean(), yerr=(s.std(ddof=1) if len(s) > 1 else 0), color=COL[m], capsize=4)
        ax.text(i, s.mean(), f"{s.mean():.3g}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(range(len(METHODS)))
    ax.set_xticklabels([NAMES[m] for m in METHODS], fontsize=8)
    ax.set_title(title)
    if log:
        ax.set_yscale("log")
plt.tight_layout()
plt.savefig(FIG / "resources.png", bbox_inches="tight")
plt.show()

## 4. Забывание общих навыков (H5)

Разница относительно исходной модели в процентных пунктах: для GSM8K это прирост (цель), для остальных обычно падение (забывание). Все метрики 0-shot, как в `configs/eval.yaml`.

In [ ]:
if len(base):
    b = base.iloc[0]
    bench_all = {"GSM8K (цель)": M_GSM, **BENCH}
    delta = pd.DataFrame({
        NAMES[m]: {name: 100 * (core[core.method == m][c].mean() - b[c])
                   for name, c in bench_all.items() if c in core and c in base}
        for m in METHODS if len(core[core.method == m])
    })
    ax = delta.plot.bar(figsize=(9, 4), color=[COL[m] for m in METHODS if NAMES[m] in delta.columns], width=0.8)
    ax.axhline(0, color="k", lw=0.8)
    ax.set(ylabel="Δ относительно исходной модели, п.п.", title="Прирост на цели и забывание общих навыков")
    ax.tick_params(axis="x", rotation=0)
    plt.tight_layout()
    plt.savefig(FIG / "forgetting.png", bbox_inches="tight")
    plt.show()
    display(delta.round(2))
else:
    print("Нет оценки исходной модели (запустите bash scripts/run.sh baseline)")

Выборки ARC, HellaSwag (`limit=2000`) и MMLU (по 50 вопросов на предмет) небольшие, поэтому разницу в десятые доли п.п. не интерпретируем.

## 5. Кривые обучения

Валидационный loss по шагам для seed 0 основных конфигураций. Расхождение train и val loss помогает увидеть переобучение (особенно у полного FT на малой выборке).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for m in METHODS:
    d = core[(core.method == m) & (core.seed == 0)]
    if d.empty:
        continue
    hp = ROOT / "results" / d.iloc[0].run / "log_history.json"
    if not hp.exists():
        continue
    h = pd.DataFrame(json.loads(hp.read_text()))
    if "loss" in h:
        t = h.dropna(subset=["loss"])
        axes[0].plot(t.step, t.loss.rolling(5, min_periods=1).mean(), color=COL[m], label=NAMES[m])
    if "eval_loss" in h:
        v = h.dropna(subset=["eval_loss"])
        axes[1].plot(v.step, v.eval_loss, "o-", color=COL[m], label=NAMES[m], ms=3)
axes[0].set(title="Train loss (скользящее среднее)", xlabel="шаг", ylabel="loss")
axes[1].set(title="Validation loss", xlabel="шаг", ylabel="loss")
axes[0].legend()
plt.tight_layout()
plt.savefig(FIG / "curves.png", bbox_inches="tight")
plt.show()

## 6. Подбор скорости обучения

Выбор по `final_val_loss` на val, один seed. Вертикальная черта это lr, записанный в `configs/method/*.yaml`. Если минимум на краю сетки, расширьте сетку. Минимум val loss не обязательно даёт максимум точности на GSM8K: это ограничение выбора, о нём нужно написать.

In [ ]:
sw = ft[(ft.dataset == "gsm8k") & ft.train_size.isna() & (ft.seed == 0) &
        ((ft.method == "full") | (ft.r == R_CORE))].dropna(subset=["final_val_loss"])
fig, ax = plt.subplots(figsize=(7, 4))
for m in METHODS:
    d = sw[sw.method == m].sort_values("lr")
    if d.empty:
        continue
    ax.plot(d.lr, d.final_val_loss, "o-", color=COL[m], label=NAMES[m])
    ax.axvline(chosen[m], color=COL[m], ls=":", alpha=0.7)
ax.set(xscale="log", xlabel="скорость обучения", ylabel="final val loss", title="Подбор lr (по val loss, seed 0)")
ax.legend()
plt.tight_layout()
plt.savefig(FIG / "lr_sweep.png", bbox_inches="tight")
plt.show()
sw[["run", "method", "lr", "final_val_loss"]].sort_values(["method", "lr"]).round(4)

## 7. Размер выборки (H3) и ранг (H4)

Для сравнения по размеру берём seed 0 (подвыборки 1k, 3k, весь train). Квадраты это MetaMathQA (25k, 1 эпоха): у него другое число эпох и другой источник, поэтому как продолжение кривой его читать нельзя, это отдельная точка.

In [ ]:
sz = ft[ft.lr_chosen & (ft.seed == 0) & ((ft.method == "full") | (ft.r == R_CORE))].dropna(subset=[M_GSM, "n_train"])
fig, ax = plt.subplots(figsize=(7, 4.4))
for m in METHODS:
    d = sz[(sz.method == m) & (sz.dataset == "gsm8k")].sort_values("n_train")
    if len(d):
        ax.plot(d.n_train, 100 * d[M_GSM], "o-", color=COL[m], label=NAMES[m])
    dm = sz[(sz.method == m) & (sz.dataset == "metamathqa")]
    if len(dm):
        ax.scatter(dm.n_train, 100 * dm[M_GSM], marker="s", facecolors="none", edgecolors=COL[m], s=70)
if len(base) and M_GSM in base:
    ax.axhline(100 * base[M_GSM].mean(), color=COL["baseline"], ls="--", lw=1, label="исходная, 0-shot")
ax.set(xscale="log", xlabel="число обучающих примеров", ylabel="GSM8K, flexible, %", title="Качество против размера выборки (seed 0)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG / "size.png", bbox_inches="tight")
plt.show()

piv = (sz[sz.dataset == "gsm8k"].pivot_table(index="n_train", columns="method", values=M_GSM) * 100).round(2)
if {"qlora", "full"} <= set(piv.columns):
    piv["QLoRA − полный FT"] = (piv["qlora"] - piv["full"]).round(2)
piv

In [ ]:
rk = full_data[(full_data.seed == 0) & full_data.method.isin(["lora", "qlora"])].dropna(subset=[M_GSM, "r"])
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for m in ["lora", "qlora"]:
    d = rk[rk.method == m].sort_values("r")
    if len(d):
        ax.plot(d.r, 100 * d[M_GSM], "o-", color=COL[m], label=NAMES[m])
fseed0 = full_data[(full_data.method == "full") & (full_data.seed == 0)][M_GSM]
if len(fseed0):
    ax.axhline(100 * fseed0.mean(), color=COL["full"], ls="--", label="полный FT (seed 0)")
ax.set(xscale="log", xlabel="ранг LoRA r", ylabel="GSM8K, flexible, %", title="Качество против ранга (seed 0)")
ax.set_xticks(sorted(rk.r.unique()) if len(rk) else [])
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG / "rank.png", bbox_inches="tight")
plt.show()

## 8. Числа для проверки гипотез

Сводка цифр для `docs/hypotheses.md`. Вывод «подтверждена / опровергнута» делаем сами, сверяя разницу с разбросом по seed и доверительным интервалом из раздела 1.

In [ ]:
def safe(f):
    try:
        return f()
    except Exception as e:
        return f"нет данных ({type(e).__name__}: {e})"

g = core.groupby("method")
acc = g[M_GSM].mean() * 100
sd = g[M_GSM].std(ddof=1) * 100
mem = g["peak_mem_allocated_gb"].mean()
thr = g["tokens_per_s"].mean()

print("H1  QLoRA − полный FT:", safe(lambda: f"{acc['qlora'] - acc['full']:+.2f} п.п. (std по seed: full {sd['full']:.2f}, qlora {sd['qlora']:.2f})"))
print("H1  LoRA − полный FT: ", safe(lambda: f"{acc['lora'] - acc['full']:+.2f} п.п."))
print("H2  память full/qlora:", safe(lambda: f"{mem['full'] / mem['qlora']:.2f}x; full/lora: {mem['full'] / mem['lora']:.2f}x"))
print("H2  скорость qlora/full:", safe(lambda: f"{thr['qlora'] / thr['full']:.2f}x (меньше 1: QLoRA медленнее)"))
print("H3  разрыв QLoRA − full по размеру выборки: см. таблицу в разделе 7")
print("H4  эффект ранга (lora, макс − мин):", safe(lambda: f"{100 * (rk[rk.method=='lora'][M_GSM].max() - rk[rk.method=='lora'][M_GSM].min()):.2f} п.п."))
print("H4  эффект квантизации (lora − qlora при r=%d):" % R_CORE, safe(lambda: f"{acc['lora'] - acc['qlora']:+.2f} п.п."))
if len(base):
    b = base.iloc[0]
    for name, c in BENCH.items():
        print(f"H5  Δ {name}:", safe(lambda: ", ".join(f"{NAMES[m]} {100*(core[core.method==m][c].mean()-b[c]):+.2f}" for m in METHODS)))

## Выводы (заполнить по результатам)

- **Качество:** разница между методами на GSM8K X п.п. при разбросе по seed Y п.п.: различима / не различима.
- **Ресурсы:** QLoRA экономит память в N раз и работает в M раз медленнее (или быстрее) полного FT.
- **Компромисс:** какой метод оптимален по соотношению «качество на единицу памяти» и когда его выбирать.
- **Забывание:** какие метрики просели сильнее у полного FT; насколько LoRA/QLoRA «забывают меньше».
- **Размер выборки и ранг:** где растёт или исчезает разрыв; влияет ли ранг сильнее квантизации.
- **Ограничения:** одна модель 0.5B, один домен, малое число seed, оценка на подмножествах ARC, HellaSwag, MMLU, подбор lr по val loss, возможное загрязнение бенчмарков.
- **Что делать дальше:** например, более крупная модель, другой домен, парные тесты на уровне задач (включить `log_samples` в eval).